# 🚀 Carnet de mission Python — Blocs 1, 2 et 3

**Ce carnet marche sans internet.** Tout est dedans : le cours, les exemples, les exercices, les corrections automatiques, les indices et les évaluations.

| Bloc | Sujet | Contenu |
|---|---|---|
| 1 | 🔁 La boucle `while` | cours, exemples, 7 exercices, évaluation |
| 2 | 📦 Les listes | cours, exemples, 7 exercices, évaluation |
| 3 | 🔤 Les chaînes de caractères | cours, exemples, 7 exercices, évaluation |

## 📖 Mode d'emploi

1. **Lance la cellule ⚙️ juste en dessous**, une seule fois. Elle doit afficher « ✅ Moteur prêt ! ». Si tu fermes l'appli, relance-la.
2. **Lis le cours** et lance les exemples ▶. Tu peux les modifier pour faire des essais.
3. **Exercices** : écris ton code dans la cellule qui commence par `%%exo` (laisse cette première ligne !), puis lance-la. Tu verras :
   - ✅ **BONNE RÉPONSE** si c'est juste,
   - ❌ **PAS ENCORE** avec une explication de ce qui ne va pas.

   Quand ton programme utilise `input()`, **c'est moi qui réponds à ta place** pour le tester. Tu n'as rien à taper.
4. **Tu bloques ?** Lance la cellule `indice(...)` sous l'exercice. Chaque fois que tu la relances, tu as un indice de plus.
5. **Vraiment bloqué ?** Après un essai, tape `solution("1.1")` dans une cellule pour voir une réponse expliquée.
6. **À la fin de chaque bloc** : un QCM et 2 exercices **sans indice**, puis `bilan(1)` te donne ta note sur 20.
7. `progression()` montre tout ce que tu as réussi. Au retour, `export()` affiche tout ton code : copie-le et envoie-le à Claude pour une correction détaillée.

## ⚠️ Spécial iPhone

- **Les guillemets** : l'iPhone transforme `"` en `“ ”`, et Python ne les comprend pas. Dans les cellules `%%exo`, je les corrige tout seul. Pour être tranquille partout : **Réglages → Général → Clavier → désactive « Ponctuation intelligente »**.
- **Les majuscules** : l'iPhone met une majuscule en début de ligne (`While` au lieu de `while`). Désactive aussi « Majuscules auto » au même endroit.
- **Boucle infinie dans une cellule normale** : appuie sur le bouton ■ (stop) de l'appli. Dans les cellules `%%exo`, je l'arrête tout seul.

In [ ]:
# ⚙️ MOTEUR DES EXERCICES — lance cette cellule EN PREMIER (tu n'as pas besoin de la lire)
# Si tu fermes l'application, relance-la avant de continuer.
import builtins, sys, json, base64, re, unicodedata

_DATA = json.loads(base64.b64decode("...").decode("utf-8"))
_FICHIER = "progression_python.json"
try:
    with open(_FICHIER, encoding="utf-8") as _f:
        _ETAT = json.load(_f)
except Exception:
    _ETAT = {}
for _k in ("essais", "reussi", "indices", "code", "solutions", "qcm", "bilans"):
    _ETAT.setdefault(_k, {})


def _sauver():
    try:
        with open(_FICHIER, "w", encoding="utf-8") as f:
            json.dump(_ETAT, f, ensure_ascii=False, indent=1)
    except Exception:
        pass


class _TropDInputs(Exception):
    pass


class _Infini(Exception):
    pass


class _Resultat:
    def __init__(self):
        self.out = ""
        self.transcript = ""
        self.ns = {}
        self.used = 0
        self.erreur = None
        self.inputs = []
        self.code = ""

    @property
    def lignes(self):
        return [l.rstrip() for l in self.out.rstrip().split("\n")] if self.out.strip() else []

    @property
    def pleines(self):
        return [l.strip() for l in self.lignes if l.strip()]


def _plain(t):
    return unicodedata.normalize("NFD", str(t)).encode("ascii", "ignore").decode().lower()


def _nums(t):
    return [float(x) if "." in x else int(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(t))]


def _dernier(t):
    n = _nums(t)
    return n[-1] if n else None


def _entiers_seuls(r):
    return [int(l) for l in r.pleines if re.fullmatch(r"-?\d+", l)]


def _nettoyer(code):
    for a, b in (("“", '"'), ("”", '"'), ("„", '"'), ("«", '"'), ("»", '"'),
                 ("‘", "'"), ("’", "'"), (" ", " "), (" ", " "), ("−", "-"),
                 ("–", "-"), ("—", "-"), ("…", "..."), ("\t", "    ")):
        code = code.replace(a, b)
    return code


def _executer(code, inputs, limite=300000):
    r = _Resultat()
    r.inputs = list(inputs)
    r.code = code
    file = list(inputs)

    def faux_input(prompt=""):
        if not file:
            raise _TropDInputs()
        v = file.pop(0)
        r.used += 1
        r.transcript += str(prompt) + str(v) + "\n"
        return str(v)

    def faux_print(*args, sep=" ", end="\n", file=None, flush=False):
        t = sep.join(str(a) for a in args) + end
        r.out += t
        r.transcript += t

    compteur = [0]

    def traceur(frame, event, arg):
        if frame.f_code.co_filename != "<ton code>":
            return None
        if event == "line":
            compteur[0] += 1
            if compteur[0] > limite:
                raise _Infini()
        return traceur

    try:
        prog = compile(code, "<ton code>", "exec")
    except SyntaxError as e:
        r.erreur = e
        return r
    ns = {"__name__": "__main__", "input": faux_input, "print": faux_print}
    ancienne = sys.gettrace()
    sys.settrace(traceur)
    try:
        exec(prog, ns)
    except (Exception, SystemExit) as e:
        r.erreur = e
    finally:
        sys.settrace(ancienne)
    r.ns = ns
    return r


def _ligne_erreur(e):
    if isinstance(e, SyntaxError):
        return e.lineno
    tb, ligne = e.__traceback__, None
    while tb is not None:
        if tb.tb_frame.f_code.co_filename == "<ton code>":
            ligne = tb.tb_lineno
        tb = tb.tb_next
    return ligne


def _explique(e, inputs):
    ligne = _ligne_erreur(e)
    ou = f" (regarde la ligne {ligne})" if ligne else ""
    nom, msg = type(e).__name__, str(e)
    if isinstance(e, _Infini):
        return "Ton programme tourne sans s'arrêter : c'est sûrement une boucle infinie 🔁. Est-ce que la variable de ta condition change bien à chaque tour ?"
    if isinstance(e, _TropDInputs):
        return ("Ton programme pose plus de questions (input) que prévu. J'ai répondu : "
                + (", ".join(f"« {x} »" for x in inputs) if inputs else "rien, car cet exercice n'a pas besoin d'input()")
                + ". Vérifie la condition de ta boucle, ou le nombre de input().")
    if isinstance(e, (IndentationError, TabError)):
        return "Problème de décalage (indentation)" + ou + " : les lignes dans une boucle ou un if sont décalées de 4 espaces, les autres non."
    if isinstance(e, SyntaxError):
        return ("Erreur de syntaxe" + ou + " : as-tu oublié les deux-points « : » à la fin du while, for ou if ? "
                "Une parenthèse ou un guillemet pas fermé ? Un « = » au lieu de « == » dans une condition ?")
    if isinstance(e, NameError):
        n = getattr(e, "name", None) or msg
        if isinstance(n, str) and n[:1].isupper() and n.lower() in ("while", "for", "if", "print", "input", "int", "else", "elif", "range", "len"):
            return f"Ton clavier a mis une majuscule à « {n} »" + ou + f". En Python, il faut l'écrire en minuscules : « {n.lower()} »."
        return f"Python ne connaît pas « {n} »" + ou + ". Vérifie l'orthographe (majuscules comprises) et que la variable est créée AVANT d'être utilisée."
    if isinstance(e, TypeError) and "str" in msg:
        return "Tu mélanges du texte et des nombres" + ou + ". Rappel : input() donne du texte. Pour un nombre, utilise int(input(...))."
    if isinstance(e, ValueError) and "int()" in msg:
        return "int() n'a pas réussi à transformer ce texte en nombre" + ou + ". As-tu mis int() sur une réponse qui n'est pas un nombre ?"
    if isinstance(e, IndexError):
        return "Tu demandes une case qui n'existe pas" + ou + ". Les indices commencent à 0, et la dernière case est à l'indice len(...) - 1."
    if isinstance(e, ZeroDivisionError):
        return "Division par zéro" + ou + " : c'est impossible, même pour Python 😅"
    if isinstance(e, AttributeError):
        return "Cette fonction n'existe pas pour ce type de donnée" + ou + f" : {msg}"
    return f"Ton programme a planté avec l'erreur {nom}" + ou + f" : {msg}"


def _affiche_transcript(r):
    if r is None:
        return
    lignes = r.transcript.rstrip("\n").split("\n") if r.transcript.strip() else []
    rep = ", ".join(f"« {x} »" for x in r.inputs)
    print("📺 Ce que fait ton programme" + (f" (j'ai répondu {rep} à ta place)" if r.inputs else "") + " :")
    if not lignes:
        print("   (rien ne s'affiche)")
    for l in lignes[:30]:
        print("   │ " + l)
    if len(lignes) > 30:
        print(f"   │ … ({len(lignes) - 30} lignes de plus)")
    print()


def _verifier(ident, code):
    ex = _EXOS.get(ident)
    if ex is None:
        print(f"❓ Je ne connais pas l'exercice « {ident} ». Vérifie le numéro après %%exo.")
        return
    code = _nettoyer(code)
    utiles = [l for l in code.split("\n") if l.strip() and not l.strip().startswith("#")]
    if len(utiles) <= ex.get("depart", 0):
        print("✍️ Écris ton code sous la ligne %%exo, puis relance la cellule ▶")
        return
    _ETAT["essais"][ident] = _ETAT["essais"].get(ident, 0) + 1
    _ETAT["code"][ident] = code
    probleme, premier, faits = None, None, 0
    maj = re.search(r"^\s*(While|For|If|Elif|Else|Print|Input|Int|Range|Len)\b", code, re.M)
    if maj:
        probleme = f"Ton clavier a mis une majuscule à « {maj.group(1)} ». En Python, il faut l'écrire en minuscules : « {maj.group(1).lower()} »."
    for motif, msg in ex.get("oblige", []):
        if not probleme and not re.search(motif, code):
            probleme = msg
    for motif, msg in ex.get("interdit", []):
        if re.search(motif, code):
            probleme = msg
    for t in ex["tests"]:
        c = code
        if "swap" in t:
            ancien, nouveau = t["swap"]
            if ancien not in c:
                continue
            c = c.replace(ancien, nouveau)
        r = _executer(c, t.get("inputs", []))
        if premier is None:
            premier = r
        if probleme:
            break
        if r.erreur is not None:
            probleme = _explique(r.erreur, t.get("inputs", [])) + ex.get("astuce", {}).get(type(r.erreur).__name__, "")
            break
        m = ex["check"](r)
        if m:
            probleme = m
            break
        faits += 1
    _affiche_transcript(premier)
    bloc = ident.split(".")[0].lstrip("E")
    if probleme is None:
        _ETAT["reussi"][ident] = True
        print("✅ BONNE RÉPONSE ! Bravo 🎉")
        if faits > 1:
            print(f"   J'ai testé ton programme {faits} fois, avec des réponses différentes : tout est bon.")
        if ex["type"] == "defi":
            print("   🔥 Défi relevé ! Tu peux être fier de toi.")
        tous = [k for k, v in _EXOS.items() if k.split(".")[0].lstrip("E") == bloc and v["type"] != "eval"]
        ok = sum(1 for k in tous if _ETAT["reussi"].get(k))
        print(f"   Progression du bloc {bloc} : {ok} / {len(tous)} exercices réussis.")
    else:
        _ETAT["reussi"].setdefault(ident, False)
        print("❌ PAS ENCORE… mais tu progresses !")
        print("   👉 " + probleme)
        if ex["type"] != "eval":
            print(f'   💡 Un indice ? Lance la cellule indice("{ident}") juste en dessous.')
    _sauver()


def indice(ident):
    """Affiche le prochain indice d'un exercice."""
    ex = _EXOS.get(ident)
    if ex is None:
        print(f"❓ Je ne connais pas l'exercice « {ident} ».")
        return
    if ex["type"] == "eval":
        print("🙅 Pas d'indice pendant l'évaluation. Tu peux le faire, j'ai confiance en toi !")
        return
    h = _DATA["indices"][ident]
    k = min(_ETAT["indices"].get(ident, 0) + 1, len(h))
    _ETAT["indices"][ident] = k
    for i in range(k):
        print(f"💡 Indice {i + 1}/{len(h)} : {h[i]}")
    if k < len(h):
        print(f"\n   (Relance cette cellule pour l'indice {k + 1}.)")
    else:
        print(f'\n   C\'était le dernier indice. Toujours bloqué ? solution("{ident}") te montre une réponse.')
    _sauver()


def solution(ident, sur=False):
    """Montre une solution expliquée (après au moins un essai)."""
    ex = _EXOS.get(ident)
    if ex is None:
        print(f"❓ Je ne connais pas l'exercice « {ident} ».")
        return
    bloc = ident.split(".")[0].lstrip("E")
    if ex["type"] == "eval" and not _ETAT["bilans"].get(bloc):
        print(f"🔒 Les solutions de l'évaluation se débloquent après bilan({bloc}).")
        return
    if not _ETAT["essais"].get(ident):
        print("🔒 Essaie d'abord au moins une fois (lance ta cellule %%exo), même si ce n'est pas juste.")
        return
    if not sur:
        print("🤔 Tu es sûr ? Chercher encore un peu, c'est comme ça qu'on progresse le plus.")
        print(f'   Si tu veux vraiment voir la réponse, lance : solution("{ident}", sur=True)')
        return
    s = _DATA["solutions"][ident]
    print("📖 Une solution possible (il y en a souvent d'autres !) :\n")
    for l in s["code"].split("\n"):
        print("    " + l)
    print("\n🧠 Explications :")
    for i, x in enumerate(s["explications"], 1):
        print(f"   {i}. {x}")
    print("\n✍️ Maintenant, efface ton code et réécris la solution SANS la regarder, puis relance ta cellule.")
    _ETAT["solutions"][ident] = True
    _sauver()


def qcm(ident, reponse):
    """Répond à une question de QCM, par exemple qcm("1.Q1", "b")."""
    q = _DATA["qcm"].get(ident)
    if q is None:
        print(f"❓ Je ne connais pas la question « {ident} ».")
        return
    rep = str(reponse).strip().lower()
    if rep not in ("a", "b", "c", "d"):
        print('✍️ Remplace le ? par ta réponse : "a", "b" ou "c", puis relance la cellule.')
        return
    premiere = ident not in _ETAT["qcm"]
    if premiere:
        _ETAT["qcm"][ident] = rep
    if rep == q["bonne"]:
        print("✅ Bonne réponse !" + ("" if premiere else " (mais c'est ta première réponse qui compte pour la note)"))
    else:
        print("❌ Ce n'est pas ça." + ("" if premiere else " (c'est ta première réponse qui compte pour la note)"))
        print(f"   La bonne réponse était « {q['bonne']} ».")
    print("   🧠 " + q["explication"])
    _sauver()


def bilan(bloc):
    """Calcule ta note sur 20 pour l'évaluation d'un bloc."""
    bloc = str(bloc)
    qs = [k for k in _DATA["qcm"] if k.split(".")[0] == bloc]
    ev = [k for k, v in _EXOS.items() if v["type"] == "eval" and k.split(".")[0].lstrip("E") == bloc]
    if not qs:
        print(f"❓ Je ne connais pas le bloc {bloc}.")
        return
    manque = [k for k in qs if k not in _ETAT["qcm"]] + [k for k in ev if not _ETAT["essais"].get(k)]
    if manque:
        print(f"✋ Pas encore ! Termine d'abord l'évaluation du bloc {bloc}. Il te reste : " + ", ".join(manque))
        return
    pts_q = sum(2 for k in qs if _ETAT["qcm"].get(k) == _DATA["qcm"][k]["bonne"])
    pts_e = sum(5 for k in ev if _ETAT["reussi"].get(k))
    note = pts_q + pts_e
    print(f"📋 BILAN DU BLOC {bloc}")
    print(f"   QCM : {pts_q} / {2 * len(qs)}")
    print(f"   Exercices : {pts_e} / {5 * len(ev)}")
    print(f"   👉 NOTE : {note} / 20")
    if note >= 18:
        print("   🏆 Excellent ! Tu maîtrises ce bloc. Direction le suivant !")
    elif note >= 14:
        print("   🚀 Très bien ! Revois juste ce que tu as raté avant de continuer.")
    elif note >= 10:
        print("   👍 C'est un bon début. Refais les exercices ratés du bloc avant de passer à la suite.")
    else:
        print("   💪 Ce bloc n'est pas encore acquis, et c'est normal. Relis le cours et refais les exercices : tu vas y arriver.")
    _ETAT["bilans"][bloc] = note
    print(f'   🔓 Les solutions de l\'évaluation sont débloquées : solution("E{bloc}.1", sur=True)')
    _sauver()


def progression():
    """Affiche tous tes exercices réussis."""
    for bloc in ("1", "2", "3"):
        ids = [k for k in _EXOS if k.split(".")[0].lstrip("E") == bloc]
        print(f"Bloc {bloc} :")
        for k in ids:
            etat = "✅" if _ETAT["reussi"].get(k) else ("❌" if _ETAT["essais"].get(k) else "⬜")
            extra = " 📖" if _ETAT["solutions"].get(k) else ""
            print(f"   {etat} {k:<6} {_EXOS[k]['titre']}{extra}")
        if bloc in _ETAT["bilans"]:
            print(f"   📋 Note de l'évaluation : {_ETAT['bilans'][bloc]} / 20")
    print("\n✅ réussi   ❌ essayé   ⬜ pas encore fait   📖 solution vue")


def export():
    """Affiche tout ton code, à copier et envoyer à Claude."""
    print("# Copie tout ce texte et envoie-le à Claude avec le mot « corrige » 🚀\n")
    for k in _EXOS:
        if k in _ETAT["code"]:
            etat = "✅" if _ETAT["reussi"].get(k) else "❌"
            vu = " 📖 solution vue" if _ETAT["solutions"].get(k) else ""
            print(f"# ===== {k} {_EXOS[k]['titre']} {etat}{vu} =====")
            print(_ETAT["code"][k].strip() + "\n")
    for b, n in _ETAT["bilans"].items():
        print(f"# Évaluation bloc {b} : {n}/20")


# ---------------------------------------------------------------- vérifications
def _suite_simple(r, attendu, mot, affiche_mot):
    nums = _entiers_seuls(r)
    if not nums:
        return f"Je ne vois pas les nombres {attendu[0]}, {attendu[1]}, … {attendu[-1]}, un par ligne."
    if nums != attendu:
        if nums[0] != attendu[0]:
            return f"Tu dois commencer à {attendu[0]}, et ton programme commence à {nums[0]}. Regarde la valeur de départ de ta variable (ou l'ordre entre print et le changement de la variable)."
        if len(nums) == len(attendu) + 1:
            return f"Ton programme affiche un nombre de trop ({nums[-1]}). Regarde ta condition : < ou <= ? > ou >= ?"
        if len(nums) == len(attendu) - 1:
            return f"Il manque le dernier nombre ({attendu[-1]}). Regarde ta condition : < ou <= ? > ou >= ?"
        return f"J'attendais {attendu} et ton programme affiche {nums}."
    fin = [l for l in r.pleines if mot in _plain(l)]
    if not fin:
        return f"Les nombres sont parfaits 👏 Il manque juste « {affiche_mot} » à la fin."
    if len(fin) > 1:
        return f"« {affiche_mot} » s'affiche {len(fin)} fois : ce print est dans la boucle. Enlève son décalage."
    if mot not in _plain(r.pleines[-1]):
        return f"« {affiche_mot} » doit s'afficher à la toute fin."
    return None


def _c_1_1(r):
    return _suite_simple(r, [1, 2, 3, 4, 5], "fini", "Fini !")


def _c_1_2(r):
    return _suite_simple(r, list(range(10, 0, -1)), "decollage", "Décollage ! 🚀")


def _c_1_3(r):
    L = [_plain(l) for l in r.pleines]
    rep = " puis ".join(f"« {x} »" for x in r.inputs)
    mauvais = sum(1 for l in L if "incorrect" in l)
    bons = sum(1 for l in L if "autoris" in l)
    if r.used < len(r.inputs):
        return f"Avec {rep}, ton programme s'arrête trop tôt : il n'a demandé le mot de passe que {r.used} fois."
    if not bons:
        return f"Avec {rep}, je ne vois pas « Accès autorisé 🚀 » à la fin."
    if bons > 1:
        return "« Accès autorisé » s'affiche plusieurs fois. Il doit s'afficher une seule fois, après la boucle."
    if mauvais != len(r.inputs) - 1:
        return f"Avec {rep}, « Mot de passe incorrect ❌ » devrait s'afficher {len(r.inputs) - 1} fois, et il s'affiche {mauvais} fois."
    if "autoris" not in L[-1]:
        return "« Accès autorisé » doit être la dernière chose affichée."
    return None


def _c_1_4(r):
    c = int(r.inputs[0])
    ws, wr = c // 7, c % 7
    nl = [l for l in r.pleines if re.search(r"\d", l)]
    if not nl:
        return f"Avec {c} litres, ton programme n'affiche aucun nombre."
    if len(nl) > 2:
        return "Ton programme affiche beaucoup de lignes avec des nombres. Les print du résultat doivent être APRÈS la boucle, sans décalage."
    n = [x for l in nl for x in _nums(l)]
    if ws in n and wr in n and (ws != wr or n.count(ws) >= 2):
        return None
    if c == 7 and 0 in n and 7 in n:
        return "Avec 7 litres pile, la fusée peut voler 1 seconde. « Au moins 7 », ça s'écrit >= et pas >."
    return f"Avec {c} litres, la fusée vole {ws} seconde(s) et il reste {wr} litre(s). Ton programme affiche les nombres {n}."


def _c_1_5(r):
    att = ["P" if int(x) > 37 else "G" for x in r.inputs[:-1]]
    seq = ["P" if "plus petit" in _plain(l) else "G" for l in r.pleines if "plus petit" in _plain(l) or "plus grand" in _plain(l)]
    if r.used < len(r.inputs):
        return f"Avec les propositions {', '.join(r.inputs)}, ton programme s'arrête avant que le joueur ait trouvé."
    for i, a in enumerate(att):
        if i >= len(seq) or seq[i] != a:
            bon = "Plus petit !" if a == "P" else "Plus grand !"
            return f"Quand le joueur propose {r.inputs[i]}, le secret (37) est {'plus petit' if a == 'P' else 'plus grand'} : il faut afficher « {bon} »."
    if len(seq) > len(att):
        return "Ton programme affiche « Plus petit » ou « Plus grand » même quand le joueur a trouvé."
    g = _dernier(r.out)
    if g != len(r.inputs):
        if g == len(r.inputs) - 1:
            return "Ton compteur oublie un essai : as-tu compté le tout premier nombre proposé ?"
        return f"Avec {len(r.inputs)} propositions, il faut afficher « Trouvé en {len(r.inputs)} essais », et ton programme affiche {g}."
    return None


def _c_1_6(r):
    lim = int(r.inputs[0])
    s = k = 0
    while s <= lim:
        k += 1
        s += k
    nl = [l for l in r.pleines if re.search(r"\d", l)]
    if not nl:
        return f"Avec la limite {lim}, ton programme n'affiche aucun nombre."
    if len(nl) > 2:
        return "Ton programme affiche beaucoup de lignes avec des nombres. Les print du résultat doivent être APRÈS la boucle."
    n = [x for l in nl for x in _nums(l)]
    n = [x for x in n if x != lim] or n
    if k in n and s in n:
        return None
    return f"Avec la limite {lim}, il faut additionner {k} nombres (1 + 2 + … + {k} = {s}). Ton programme affiche {n}."


def _syracuse(n):
    s = [n]
    while n != 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        s.append(n)
    return s


def _c_1_7(r):
    d = int(r.inputs[0])
    seq = _syracuse(d)
    if re.search(r"\d\.0\b", r.out):
        return "Je vois des nombres avec « .0 » : utilise // au lieu de / pour diviser sans virgule."
    g = _entiers_seuls(r)
    if g != seq:
        if g == seq[1:]:
            return f"Avec {d}, il manque le nombre de départ : affiche-le avant la boucle."
        if g == seq[:-1]:
            return f"Avec {d}, la suite doit se terminer par 1 : affiche-le aussi."
        k = next((i for i in range(len(seq)) if i >= len(g) or g[i] != seq[i]), len(seq))
        if k < len(seq):
            return f"Avec {d}, le nombre n°{k + 1} devrait être {seq[k]}" + (f", et ton programme affiche {g[k]}." if k < len(g) else ".")
        return f"Avec {d}, ta suite est trop longue : elle doit s'arrêter à 1."
    if _dernier(r.out) != len(seq) - 1:
        return f"La suite est parfaite 👏 Avec {d}, il y a {len(seq) - 1} étapes, et ton programme affiche {_dernier(r.out)} à la fin."
    return None


def _c_E1_1(r):
    n = int(r.inputs[0])
    att = list(range(3, n + 1, 3))
    g = _entiers_seuls(r)
    if g != att:
        return f"Avec n = {n}, j'attendais {att if att else 'aucun nombre'} (un par ligne), et ton programme affiche {g if g else 'aucun nombre'}."
    return None


def _c_E1_2(r):
    p, o = int(r.inputs[0]), int(r.inputs[1])
    h = 0
    while p < o:
        p *= 2
        h += 1
    g = _dernier(r.out)
    if g != h:
        return f"Avec une population de {r.inputs[0]} et un objectif de {r.inputs[1]}, il faut {h} heure(s). Ton programme affiche {g}."
    return None


def _c_2_1(r):
    p = r.ns.get("planetes")
    if not isinstance(p, list):
        return "Je ne trouve pas de liste qui s'appelle planetes."
    if [_plain(x) for x in p] != ["mercure", "venus", "terre", "mars"]:
        return f"Ta liste planetes contient {p}. Elle doit contenir Mercure, Vénus, Terre, Mars, dans cet ordre."
    if not r.pleines or _plain(r.pleines[-1]) != "terre":
        return "Ta liste est parfaite 👏 Il faut maintenant afficher la 3e planète (Terre)."
    return None


def _c_2_2(r):
    notes = r.ns.get("notes")
    if not isinstance(notes, list) or not notes:
        return "Je ne trouve plus la liste notes. Ne modifie pas la première ligne."
    mx, mn, moy = max(notes), min(notes), sum(notes) / len(notes)
    n = _nums(r.out)
    if mx not in n:
        return f"Avec les notes {notes}, je ne vois pas la meilleure note ({mx})."
    if mn not in n:
        return f"Avec les notes {notes}, je ne vois pas la moins bonne note ({mn})."
    if not any(abs(x - moy) < 0.06 for x in n):
        return f"Avec les notes {notes}, la moyenne est {round(moy, 2)} et je ne la vois pas. La moyenne, c'est la somme divisée par le nombre de notes."
    return None


def _c_2_3(r):
    d = r.ns.get("decompte")
    if not isinstance(d, list):
        return "Je ne trouve pas de liste qui s'appelle decompte."
    if d != list(range(10, 0, -1)):
        return f"Ta liste decompte vaut {d}. Elle doit valoir [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]."
    if "[10, 9" not in r.out:
        return "Ta liste est parfaite 👏 Il faut aussi l'afficher avec print(decompte)."
    return None


def _c_2_4(r):
    b = r.ns.get("bagages")
    if not isinstance(b, list):
        return "Je ne trouve pas de liste qui s'appelle bagages."
    if b != r.inputs:
        return f"J'ai répondu {r.inputs}, et ta liste bagages vaut {b}."
    if not any("3" in l for l in r.pleines):
        return "Je ne vois pas la phrase « Tu emportes 3 objets : »."
    for o in r.inputs:
        if not any(l.startswith("-") and o in l for l in r.pleines):
            return f"Je ne vois pas la ligne « - {o} ». Chaque objet doit s'afficher sur sa ligne, après un tiret."
    return None


def _c_2_5(r):
    t = r.ns.get("temperatures")
    if not isinstance(t, list):
        return "Je ne trouve plus la liste temperatures. Ne modifie pas la première ligne."
    pos, neg = [x for x in t if x > 0], sum(1 for x in t if x < 0)
    p = r.ns.get("positives")
    if p != pos:
        return f"Avec {t}, la liste positives doit valoir {pos}, et la tienne vaut {p}." + (" Attention : 0 n'est pas positif !" if isinstance(p, list) and 0 in p else "")
    if str(pos) not in r.out:
        return "Ta liste positives est juste 👏 Affiche-la avec print(positives)."
    autres = [x for l in r.pleines if "[" not in l for x in _nums(l)]
    if not autres or autres[-1] != neg:
        return f"Avec {t}, il y a {neg} température(s) négative(s) (0 ne compte pas !). Ton programme affiche {autres[-1] if autres else 'rien'}."
    return None


def _c_2_6(r):
    a = r.ns.get("altitudes")
    if not isinstance(a, list) or not a:
        return "Je ne trouve plus la liste altitudes. Ne modifie pas la première ligne."
    g = _dernier(r.out)
    if g != max(a):
        if g == 0 and max(a) < 0:
            return f"Avec {a} (que des nombres négatifs), ton programme trouve 0. Astuce : commence avec la première valeur de la liste, pas avec 0."
        return f"Avec {a}, la plus grande altitude est {max(a)}, et ton programme affiche {g}."
    return None


def _fibo(n):
    f, a, b = [], 1, 1
    while len(f) < n:
        f.append(a)
        a, b = b, a + b
    return f


def _c_2_7(r):
    n = int(r.inputs[0])
    f = r.ns.get("fib")
    if not isinstance(f, list):
        return "Je ne trouve pas de liste qui s'appelle fib."
    if f != _fibo(n):
        return f"Avec n = {n}, fib doit valoir {_fibo(n)}, et la tienne vaut {f}."
    return None


def _c_E2_1(r):
    s = r.ns.get("scores")
    if not isinstance(s, list):
        return "Je ne trouve plus la liste scores. Ne modifie pas la première ligne."
    w = sum(1 for x in s if x >= 1000)
    g = _nums(r.out)
    if w not in g:
        return f"Avec {s}, il y a {w} score(s) supérieur(s) ou égal(aux) à 1000. Ton programme affiche {g}."
    return None


def _c_E2_2(r):
    n = int(r.inputs[0])
    m = r.ns.get("multiples")
    w = [7 * i for i in range(1, n + 1)]
    if m != w:
        return f"Avec n = {n}, multiples doit valoir {w}, et la tienne vaut {m}."
    return None


def _c_3_1(r):
    p = r.inputs[0]
    if not any(p in l and len(p) in _nums(l) for l in r.pleines):
        if not any(p in l for l in r.pleines):
            return f"J'ai répondu « {p} », et je ne vois pas ce prénom dans ce que tu affiches."
        return f"Avec « {p} », je ne vois pas le nombre de lettres ({len(p)})."
    return None


def _c_3_2(r):
    m = r.inputs[0]
    L = r.pleines
    if len(L) < 2:
        return f"Avec « {m} », il faut 2 lignes : la première lettre, puis la dernière."
    if not L[-2].endswith(m[0]) or not L[-1].endswith(m[-1]):
        return f"Avec « {m} », il faut afficher « {m[0]} » puis « {m[-1]} ». Ton programme affiche « {L[-2]} » puis « {L[-1]} »."
    return None


def _c_3_3(r):
    m = r.inputs[0]
    w = sum(1 for c in m if c in "aeiouy")
    g = _dernier(r.out)
    if len([l for l in r.pleines if re.search(r"\d", l)]) > 1:
        return "Ton programme affiche plusieurs nombres. Le print final doit être APRÈS la boucle, sans décalage."
    if g != w:
        if g == len(m):
            return f"Avec « {m} », tu trouves {g} : c'est le nombre de lettres ! Le « +1 » doit être dans le if."
        return f"Avec « {m} », il y a {w} voyelle(s) et ton programme en compte {g}."
    return None


def _c_3_4(r):
    p = r.inputs[0]
    L = r.pleines
    if p.upper() not in L:
        return f"Avec « {p} », je ne vois pas la ligne « {p.upper()} »."
    if p.upper().replace(" ", "_") not in L:
        return f"Avec « {p} », je ne vois pas la ligne « {p.upper().replace(' ', '_')} »."
    return None


def _c_3_5(r):
    m = r.inputs[0]
    pal = m == m[::-1]
    if not r.pleines or "palindrome" not in _plain(r.pleines[-1]):
        return f"Avec « {m} », je ne vois pas de phrase avec le mot « palindrome » à la fin."
    dit_non = "pas" in _plain(r.pleines[-1])
    if dit_non == pal:
        return f"« {m} » " + ("EST un palindrome" if pal else "n'est PAS un palindrome") + ", et ton programme dit le contraire."
    return None


def _c_3_6(r):
    m = r.inputs[0]
    if not r.pleines or r.pleines[-1] != m[::-1]:
        return f"Avec « {m} », il faut afficher « {m[::-1]} »" + (f", et ton programme affiche « {r.pleines[-1]} »." if r.pleines else ".")
    return None


def _cesar(t, d):
    a = "abcdefghijklmnopqrstuvwxyz"
    return "".join(a[(a.index(c) + d) % 26] if c in a else c for c in t)


def _c_3_7(r):
    t, d = r.inputs[0], int(r.inputs[1])
    w = _cesar(t, d)
    if not r.lignes or r.lignes[-1].strip() != w:
        dern = r.lignes[-1].strip() if r.lignes else ""
        msg = f"Avec « {t} » et un décalage de {d}, il faut afficher « {w} »" + (f", et ton programme affiche « {dern} »." if dern else ".")
        if "z" in t + w and dern and len(dern) == len(w):
            msg += " Après z, on revient à a : pense à % 26."
        return msg
    return None


def _c_E3_1(r):
    m = r.inputs[0]
    if _dernier(r.out) != m.count("e"):
        return f"Avec « {m} », il y a {m.count('e')} « e », et ton programme affiche {_dernier(r.out)}."
    return None


def _c_E3_2(r):
    m = r.inputs[0]
    w = "-".join(m)
    if not r.pleines or r.pleines[-1] != w:
        return f"Avec « {m} », il faut afficher « {w} »" + (f", et ton programme affiche « {r.pleines[-1]} »." if r.pleines else ".")
    return None


_W = [(r"\bwhile\b", "Cet exercice doit utiliser une boucle while. Je n'en vois pas dans ton code.")]
_EXOS = {
    "1.1": dict(titre="Compter de 1 à 5", type="debut", tests=[{}], check=_c_1_1, oblige=_W),
    "1.2": dict(titre="Le compte à rebours 🚀", type="debut", tests=[{}], check=_c_1_2, oblige=_W),
    "1.3": dict(titre="Le mot de passe 🔐", type="debut", check=_c_1_3, oblige=_W,
                tests=[{"inputs": ["fusee", "mit", "polytechnique"]}, {"inputs": ["polytechnique"]}, {"inputs": ["Polytechnique", "polytechnique"]}]),
    "1.4": dict(titre="Le réservoir ⛽", type="debut", check=_c_1_4, oblige=_W,
                tests=[{"inputs": ["50"]}, {"inputs": ["7"]}, {"inputs": ["3"]}, {"inputs": ["100"]}]),
    "1.5": dict(titre="Le nombre secret 🎮", type="fin", check=_c_1_5,
                oblige=_W + [(r"secret\s*=\s*37", "Pour la vérification, écris secret = 37 au début de ton programme.")],
                tests=[{"inputs": ["50", "20", "37"]}, {"inputs": ["37"]}, {"inputs": ["40", "30", "36", "38", "37"]}]),
    "1.6": dict(titre="Dépasser la limite ➕", type="fin", check=_c_1_6, oblige=_W,
                tests=[{"inputs": ["10"]}, {"inputs": ["100"]}, {"inputs": ["0"]}]),
    "1.7": dict(titre="🔥 Le mystère de Syracuse", type="defi", check=_c_1_7, oblige=_W,
                tests=[{"inputs": ["6"]}, {"inputs": ["1"]}, {"inputs": ["27"]}]),
    "E1.1": dict(titre="Les multiples de 3", type="eval", check=_c_E1_1, oblige=_W,
                 tests=[{"inputs": ["10"]}, {"inputs": ["2"]}, {"inputs": ["12"]}]),
    "E1.2": dict(titre="Les bactéries 🦠", type="eval", check=_c_E1_2, oblige=_W,
                 tests=[{"inputs": ["5", "100"]}, {"inputs": ["1", "1"]}, {"inputs": ["2", "1000"]}]),
    "2.1": dict(titre="Les planètes 🪐", type="debut", tests=[{}], check=_c_2_1,
                oblige=[(r"planetes\s*\[\s*2\s*\]", "Affiche la 3e planète avec planetes[2] (les indices commencent à 0).")]),
    "2.2": dict(titre="Le bulletin de notes 📊", type="debut", check=_c_2_2, depart=1,
                tests=[{}, {"swap": ("[12, 15, 8, 17, 10]", "[9, 20, 14]")}, {"swap": ("[12, 15, 8, 17, 10]", "[11]")}]),
    "2.3": dict(titre="Le compte à rebours en liste", type="debut", tests=[{}], check=_c_2_3,
                oblige=[(r"\.append\s*\(", "Construis la liste avec .append() dans une boucle (pas en l'écrivant à la main 😉)."),
                        (r"\b(for|while)\b", "Utilise une boucle (for ou while) pour remplir la liste.")]),
    "2.4": dict(titre="Les bagages 🎒", type="debut", check=_c_2_4,
                oblige=[(r"\.append\s*\(", "Ajoute chaque objet à la liste avec bagages.append(...).")],
                tests=[{"inputs": ["casque", "eau", "carte"]}, {"inputs": ["fusée", "chat", "livre"]}]),
    "2.5": dict(titre="Les températures 🌡️", type="fin", check=_c_2_5, depart=1,
                tests=[{}, {"swap": ("[-5, 3, -12, 8, 0, 15, -1]", "[4, -2, 0, -7, 9]")}]),
    "2.6": dict(titre="La plus haute altitude ⛰️", type="fin", check=_c_2_6, depart=1,
                interdit=[(r"\bmax\s*\(", "Pas le droit à max() dans cet exercice : trouve la plus grande valeur avec une boucle 😉"),
                          (r"sort", "Pas le droit de trier la liste dans cet exercice : utilise une boucle 😉")],
                tests=[{}, {"swap": ("[120, 340, 90, 560, 410]", "[5, 80, 12, 99, 3]")}, {"swap": ("[120, 340, 90, 560, 410]", "[-3, -8, -1]")}]),
    "2.7": dict(titre="🔥 La suite de Fibonacci", type="defi", check=_c_2_7,
                tests=[{"inputs": ["10"]}, {"inputs": ["1"]}, {"inputs": ["2"]}]),
    "E2.1": dict(titre="Les meilleurs scores 🏆", type="eval", check=_c_E2_1, depart=1,
                 tests=[{}, {"swap": ("[450, 1200, 300, 980, 1500]", "[1000, 999, 2000]")}, {"swap": ("[450, 1200, 300, 980, 1500]", "[10, 20]")}]),
    "E2.2": dict(titre="Les multiples de 7", type="eval", check=_c_E2_2,
                 tests=[{"inputs": ["5"]}, {"inputs": ["1"]}, {"inputs": ["0"]}]),
    "3.1": dict(titre="Bonjour ! 👋", type="debut", check=_c_3_1, tests=[{"inputs": ["Edouard"]}, {"inputs": ["Ana"]}]),
    "3.2": dict(titre="Première et dernière lettre", type="debut", check=_c_3_2,
                tests=[{"inputs": ["fusee"]}, {"inputs": ["X"]}, {"inputs": ["polytechnique"]}]),
    "3.3": dict(titre="Compter les voyelles 🔤", type="debut", check=_c_3_3,
                tests=[{"inputs": ["fusee"]}, {"inputs": ["polytechnique"]}, {"inputs": ["Python"]}]),
    "3.4": dict(titre="Le crieur spatial 📢", type="debut", check=_c_3_4,
                tests=[{"inputs": ["bonjour la terre"]}, {"inputs": ["vive le mit"]}]),
    "3.5": dict(titre="Les palindromes 🔁", type="fin", check=_c_3_5,
                tests=[{"inputs": ["kayak"]}, {"inputs": ["fusee"]}, {"inputs": ["radar"]}, {"inputs": ["mit"]}]),
    "3.6": dict(titre="Le miroir 🪞", type="fin", check=_c_3_6,
                interdit=[(r"\[\s*:\s*:\s*-1\s*\]", "Pas le droit à [::-1] dans cet exercice : construis le mot à l'envers avec une boucle 😉"),
                          (r"revers", "Pas le droit à reversed() ou reverse() ici : utilise une boucle 😉")],
                tests=[{"inputs": ["fusee"]}, {"inputs": ["mit"]}, {"inputs": ["a"]}]),
    "3.7": dict(titre="🔥 Le code de César", type="defi", check=_c_3_7,
                astuce={"IndexError": " Ici, c'est sûrement quand tu dépasses le z : pense à % 26 pour revenir au début de l'alphabet."},
                tests=[{"inputs": ["abc", "1"]}, {"inputs": ["xyz", "3"]}, {"inputs": ["salut la terre", "2"]}]),
    "E3.1": dict(titre="Compter les e", type="eval", check=_c_E3_1,
                 tests=[{"inputs": ["fusee"]}, {"inputs": ["terre"]}, {"inputs": ["mars"]}]),
    "E3.2": dict(titre="Les tirets", type="eval", check=_c_E3_2,
                 tests=[{"inputs": ["mars"]}, {"inputs": ["a"]}, {"inputs": ["mit"]}]),
}


# ---------------------------------------------------------------- magies Jupyter
def _ralenti(line, cell):
    code = _nettoyer(cell)
    src = code.split("\n")
    maxi = int(line) if line.strip().isdigit() else 80
    try:
        prog = compile(code, "<ton code>", "exec")
    except SyntaxError as e:
        print("❌ " + _explique(e, []))
        return
    etapes = [0]

    def montre(*a, sep=" ", end="\n", **k):
        builtins.print("      📺 affiche :", sep.join(str(x) for x in a))

    ns = {"__name__": "__main__", "print": montre}

    def tr(frame, event, arg):
        if frame.f_code.co_filename != "<ton code>":
            return None
        if event == "line":
            etapes[0] += 1
            if etapes[0] > maxi:
                raise _Infini()
            vs = {k: v for k, v in frame.f_locals.items()
                  if not k.startswith("__") and k not in ("print",) and not callable(v) and type(v).__name__ != "module"}
            vt = ", ".join(f"{k} = {v!r}" for k, v in vs.items()) or "aucune variable"
            builtins.print(f"👉 ligne {frame.f_lineno:>2} │ {src[frame.f_lineno - 1].strip():<32} │ {vt}")
        return tr

    print("🐢 Ton code au ralenti (les variables sont montrées AVANT que la ligne s'exécute) :\n")
    ancienne = sys.gettrace()
    sys.settrace(tr)
    try:
        exec(prog, ns)
        print("\n🏁 Fin du programme.")
    except _Infini:
        print(f"\n⏹️ J'arrête après {maxi} étapes (boucle trop longue ou infinie).")
    except Exception as e:
        print("\n❌ " + _explique(e, []))
    finally:
        sys.settrace(ancienne)


try:
    from IPython.core.magic import register_cell_magic
    register_cell_magic("exo")(lambda line, cell: _verifier(line.strip(), cell))
    register_cell_magic("ralenti")(_ralenti)
    print("✅ Moteur prêt ! Tu peux commencer. 🚀")
    if any(_ETAT["essais"].values()):
        print("   (J'ai retrouvé ta progression précédente : lance progression() pour la voir.)")
except Exception as _e:
    print("⚠️ Je n'arrive pas à créer les cellules %%exo dans cette application :", _e)

---
# 🔁 BLOC 1 — La boucle `while`

## 1. À quoi ça sert ?

Avec `for`, on sait **à l'avance** combien de tours on va faire. Mais parfois, on ne le sait pas :

- **Tant que** le mot de passe est faux, on le redemande.
- **Tant qu'**il reste du carburant, la fusée vole.
- **Tant que** le joueur n'a pas trouvé le nombre secret, le jeu continue.

`while` veut dire **« tant que »** en anglais. On répète **tant que** la condition est vraie.

## 2. Comment ça marche ?

```python
compte = 3
while compte > 0:
    print(compte)
    compte = compte - 1
print("Décollage !")
```
Ce qui s'affiche :
```
3
2
1
Décollage !
```

Une boucle `while` a toujours besoin de **3 ingrédients** :

1. **Une variable créée avant la boucle** : `compte = 3`
2. **Une condition** après `while` : `compte > 0`. Tant qu'elle est vraie, on refait un tour.
3. **Une ligne qui change la variable** dans la boucle : `compte = compte - 1`. Sans elle, la boucle ne s'arrête jamais !

Et comme pour `for` et `if` : les **deux-points** `:` à la fin de la ligne, et les lignes de la boucle **décalées de 4 espaces**.

## 3. Les conditions

| Écriture | Veut dire |
|---|---|
| `x > 5` | x est plus grand que 5 |
| `x < 5` | x est plus petit que 5 |
| `x >= 5` | x est plus grand **ou égal** à 5 |
| `x <= 5` | x est plus petit **ou égal** à 5 |
| `x == 5` | x est **égal** à 5 (deux signes `=` !) |
| `x != 5` | x est **différent** de 5 |

## 4. `for` ou `while` ?

| Situation | Boucle |
|---|---|
| Afficher la table de 7 (10 lignes) | `for` : on connaît le nombre de tours |
| Redemander tant que c'est faux | `while` : on ne sait pas combien d'essais |
| Voler tant qu'il reste du carburant | `while` |
| Compter de 10 à 1 | les deux marchent ! |

✏️ **Raccourcis** : `compte -= 1` veut dire `compte = compte - 1`, et `total += 5` veut dire `total = total + 5`.

## 5. 🐢 La boucle au ralenti

La cellule ci-dessous commence par `%%ralenti` : elle exécute le code **étape par étape** et te montre la valeur des variables à chaque ligne. Lance-la et lis bien ! Tu peux mettre `%%ralenti` au-dessus de **n'importe quel code** pour le comprendre.

In [ ]:
%%ralenti
compte = 3
while compte > 0:
    print(compte)
    compte = compte - 1
print("Décollage !")

Tu vois ? La ligne 2 (`while compte > 0`) revient **à chaque tour**. Quand `compte` vaut 0, la condition est fausse : Python saute directement à la ligne 5.

## 6. Exemples à lancer ▶

**Exemple A : le compte à rebours.** Change le 5 en 8 et relance.

In [ ]:
compte = 5
while compte > 0:
    print(compte)
    compte = compte - 1
print("Décollage ! 🚀")

**Exemple B : attendre le bon ordre.** Ici, c'est toi qui tapes : écris `go` pour sortir de la boucle.

In [ ]:
ordre = ""
while ordre != "go":
    ordre = input("Tape go pour lancer : ")
print("Lancement !")

**Exemple C : doubler jusqu'à dépasser 1000.** Ici, on ne sait pas à l'avance combien de tours il faudra : c'est le travail de `while`.

In [ ]:
altitude = 1
secondes = 0
while altitude < 1000:
    altitude = altitude * 2
    secondes = secondes + 1
print("1000 m dépassés en", secondes, "secondes")
print("Altitude :", altitude, "m")

## 7. ⚠️ Les erreurs classiques

- Oublier de changer la variable dans la boucle → **boucle infinie**
- Oublier de créer la variable avant la boucle → `NameError`
- Écrire `=` au lieu de `==` dans la condition → `SyntaxError`
- Se tromper entre `>` et `>=` → un tour de trop ou de moins

---
## ✍️ Exercices du bloc 1 — on commence doucement

### 🎯 Exercice 1.1 ⭐ — Compter de 1 à 5

Avec une boucle `while`, affiche les nombres de **1 à 5** (un par ligne), puis `Fini !`.

**Résultat attendu :**
```
1
2
3
4
5
Fini !
```

In [ ]:
%%exo 1.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("1.1")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 1.2 ⭐ — Le compte à rebours 🚀

Avec une boucle `while`, affiche les nombres de **10 à 1** (un par ligne), puis `Décollage ! 🚀`.

**Résultat attendu :**
```
10
9
8
...
1
Décollage ! 🚀
```

In [ ]:
%%exo 1.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("1.2")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 1.3 ⭐⭐ — Le mot de passe 🔐

Demande le mot de passe. **Tant qu'**il n'est pas égal à `polytechnique`, affiche `Mot de passe incorrect ❌` et redemande-le. Quand il est bon, affiche `Accès autorisé 🚀`.

**Résultat attendu :**
```
Mot de passe ? fusee
Mot de passe incorrect ❌
Mot de passe ? mit
Mot de passe incorrect ❌
Mot de passe ? polytechnique
Accès autorisé 🚀
```

In [ ]:
%%exo 1.3
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("1.3")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 1.4 ⭐⭐ — Le réservoir ⛽

La fusée consomme **7 litres par seconde**. Elle vole tant qu'il reste **au moins 7 litres**. Demande combien de litres il y a au départ, puis affiche combien de secondes elle vole et combien de litres il reste.

**Résultat attendu :**
```
Carburant au départ ? 50
La fusée vole 7 secondes
Il reste 1 litres
```

In [ ]:
%%exo 1.4
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("1.4")   # 💡 lance cette cellule si tu bloques

## 💪 Exercices de fin de bloc 1 — moins d'indices, à toi de jouer !

### 🎯 Exercice 1.5 ⭐⭐⭐ — Le nombre secret 🎮

Écris `secret = 37` au début. Le joueur propose des nombres jusqu'à trouver le secret. Si sa proposition est trop grande, affiche `Plus petit !`. Si elle est trop petite, affiche `Plus grand !`. Quand il trouve, affiche `Trouvé en … essais`.

🎲 **Bonus** (dans une cellule normale) : remplace 37 par `random.randint(1, 100)` (avec `import random` au début) et joue pour de vrai !

**Résultat attendu :**
```
Ton nombre ? 50
Plus petit !
Ton nombre ? 20
Plus grand !
Ton nombre ? 37
Trouvé en 3 essais
```

In [ ]:
%%exo 1.5
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("1.5")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 1.6 ⭐⭐⭐ — Dépasser la limite ➕

Demande une limite. Additionne `1 + 2 + 3 + …` **tant que** la somme ne dépasse pas la limite. Affiche combien de nombres tu as additionnés et la somme obtenue (la première qui dépasse la limite).

**Résultat attendu :**
```
Limite ? 10
Il faut 5 nombres
La somme vaut 15
```

In [ ]:
%%exo 1.6
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("1.6")   # 💡 lance cette cellule si tu bloques

## 🔥 Défi du bloc 1

### 🎯 Exercice 1.7 🔥 — Le mystère de Syracuse

Un vrai mystère des maths ! Prends un nombre `n`. S'il est **pair**, divise-le par 2. S'il est **impair**, multiplie-le par 3 et ajoute 1. Recommence **tant que** n n'est pas égal à 1. Personne au monde n'a encore prouvé qu'on arrive toujours à 1 !

Demande `n`, affiche chaque nombre de la suite (le nombre de départ compris, un par ligne), puis le nombre d'étapes. Essaie ensuite avec 27 dans une cellule normale 😮

**Résultat attendu :**
```
Nombre de départ ? 6
6
3
10
5
16
8
4
2
1
Arrivé à 1 en 8 étapes
```

In [ ]:
%%exo 1.7
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("1.7")   # 💡 lance cette cellule si tu bloques

---
## 📝 ÉVALUATION DU BLOC 1

Pas d'indice ici : c'est le moment de voir ce que tu as retenu ! **Ta première réponse au QCM compte pour la note.**

### Partie 1 : QCM (10 points)

**Q1.** Que fait `while x < 5:` ?

- **a)** Ça répète exactement 5 fois
- **b)** Ça répète tant que x est plus petit que 5
- **c)** Ça répète jusqu'à ce que x soit plus petit que 5

In [ ]:
qcm("1.Q1", "?")   # ✍️ remplace ? par a, b ou c

**Q2.** Qu'affiche ce code ?
```python
x = 0
while x < 3:
    print(x)
    x = x + 1
```

- **a)** 1 2 3
- **b)** 0 1 2
- **c)** 0 1 2 3

In [ ]:
qcm("1.Q2", "?")   # ✍️ remplace ? par a, b ou c

**Q3.** Qu'est-ce qui provoque une boucle infinie ?

- **a)** Oublier les deux-points :
- **b)** Ne jamais changer la variable de la condition
- **c)** Mettre un print dans la boucle

In [ ]:
qcm("1.Q3", "?")   # ✍️ remplace ? par a, b ou c

**Q4.** Comment écrit-on « différent de » en Python ?

- **a)** <>
- **b)** =!
- **c)** !=

In [ ]:
qcm("1.Q4", "?")   # ✍️ remplace ? par a, b ou c

**Q5.** Quelle est la valeur de x à la fin ?
```python
x = 10
while x > 0:
    x = x - 3
```

- **a)** 0
- **b)** 1
- **c)** -2

In [ ]:
qcm("1.Q5", "?")   # ✍️ remplace ? par a, b ou c

### Partie 2 : exercices (10 points)

### 🎯 Exercice E1.1  — Les multiples de 3

Avec une boucle `while`, demande un nombre `n`, puis affiche tous les multiples de 3 **jusqu'à n compris** (un par ligne).

**Résultat attendu :**
```
n ? 10
3
6
9
```

In [ ]:
%%exo E1.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 🎯 Exercice E1.2  — Les bactéries 🦠

Une population de bactéries **double chaque heure**. Demande la population de départ, puis l'objectif. Affiche le nombre d'heures qu'il faut pour **atteindre ou dépasser** l'objectif.

**Résultat attendu :**
```
Population de départ ? 5
Objectif ? 100
Il faut 5 heures
```

In [ ]:
%%exo E1.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 📋 Ta note du bloc 1

In [ ]:
bilan(1)

---
# 📦 BLOC 2 — Les listes

## 1. Une liste, c'est quoi ?

Une liste range **plusieurs valeurs dans une seule variable**. On l'écrit entre **crochets** `[ ]`, avec des **virgules** :

```python
planetes = ["Mercure", "Vénus", "Terre", "Mars"]
notes = [12, 15, 8, 17]
vide = []
```

## 2. Les indices (les numéros des cases)

⚠️ **On commence à compter à 0 !**

| Indice | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| `planetes[...]` | Mercure | Vénus | Terre | Mars |

- `planetes[0]` → `"Mercure"`
- `planetes[2]` → `"Terre"`
- `planetes[-1]` → `"Mars"` (−1 = la dernière case)
- `planetes[4]` → **erreur** `IndexError` : la case n'existe pas !

`len(planetes)` donne le **nombre** de cases : ici 4. La dernière case est donc à l'indice `len(planetes) - 1`.

## 3. Modifier une liste

```python
planetes[1] = "Venus"          # remplace la case 1
planetes.append("Jupiter")     # ajoute à la fin
planetes.remove("Mercure")     # retire cette valeur
```

## 4. Parcourir une liste avec `for`

C'est la façon **la plus simple** d'utiliser `for` (promis, c'est plus facile que `range`) :

```python
for planete in planetes:
    print(planete)
```
À chaque tour, la variable `planete` prend la valeur de la case suivante. Pas d'indice, pas de calcul !

## 5. Les fonctions toutes prêtes

| Écriture | Résultat avec `[12, 15, 8]` |
|---|---|
| `len(notes)` | 3 (nombre de cases) |
| `sum(notes)` | 35 (la somme) |
| `max(notes)` | 15 (la plus grande) |
| `min(notes)` | 8 (la plus petite) |
| `sorted(notes)` | `[8, 12, 15]` (une copie triée) |
| `15 in notes` | `True` (15 est-il dedans ?) |

## 6. ⭐ La technique la plus importante : construire une liste

On part d'une liste **vide**, et on la remplit dans une boucle avec `append` :

```python
carres = []
for n in range(1, 6):
    carres.append(n * n)
print(carres)       # [1, 4, 9, 16, 25]
```

Regarde-la au ralenti 🐢 :

In [ ]:
%%ralenti
carres = []
n = 1
while n <= 4:
    carres.append(n * n)
    n = n + 1
print(carres)

## 7. Exemples à lancer ▶

**Exemple A : lire une liste.**

In [ ]:
planetes = ["Mercure", "Vénus", "Terre", "Mars"]
print(planetes)
print("Première :", planetes[0])
print("Dernière :", planetes[-1])
print("Nombre de planètes :", len(planetes))

**Exemple B : ajouter et parcourir.**

In [ ]:
bagages = ["casque", "eau"]
bagages.append("carte des étoiles")
for objet in bagages:
    print("-", objet)

**Exemple C : les fonctions toutes prêtes.**

In [ ]:
notes = [12, 15, 8, 17, 10]
print("Somme :", sum(notes))
print("Meilleure :", max(notes))
print("Triées :", sorted(notes))
print("17 est dedans ?", 17 in notes)

**Exemple D : compter dans une liste.** Combien de notes au-dessus de 10 ?

In [ ]:
notes = [12, 15, 8, 17, 10]
compteur = 0
for note in notes:
    if note > 10:
        compteur = compteur + 1
print(compteur, "notes au-dessus de 10")

## 8. ⚠️ Les erreurs classiques

- Oublier que les indices commencent à **0** → on prend la mauvaise case
- Demander une case qui n'existe pas → `IndexError`
- Écrire `liste.append = 5` au lieu de `liste.append(5)`
- Créer la liste vide **dans** la boucle → elle est remise à zéro à chaque tour

---
## ✍️ Exercices du bloc 2 — on commence doucement

### 🎯 Exercice 2.1 ⭐ — Les planètes 🪐

Crée une liste `planetes` contenant `"Mercure"`, `"Vénus"`, `"Terre"`, `"Mars"` (dans cet ordre), puis affiche la **3e** planète en utilisant son indice.

**Résultat attendu :**
```
Terre
```

In [ ]:
%%exo 2.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("2.1")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 2.2 ⭐ — Le bulletin de notes 📊

La liste `notes` est déjà écrite (**ne modifie pas cette ligne**, je vais la changer pour tester). Affiche la meilleure note, la moins bonne, puis la moyenne.

**Résultat attendu :**
```
Meilleure note : 17
Moins bonne note : 8
Moyenne : 12.4
```

In [ ]:
%%exo 2.2
notes = [12, 15, 8, 17, 10]
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("2.2")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 2.3 ⭐⭐ — Le compte à rebours en liste

Construis une liste `decompte` qui contient `10, 9, 8, … 1` **avec une boucle** et `.append()`, puis affiche-la.

**Résultat attendu :**
```
[10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
```

In [ ]:
%%exo 2.3
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("2.3")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 2.4 ⭐⭐ — Les bagages 🎒

Demande **3 objets** à emporter dans la fusée (3 `input`), range-les dans une liste `bagages`, puis affiche `Tu emportes 3 objets :` et chaque objet sur une ligne, après un tiret.

**Résultat attendu :**
```
Objet ? casque
Objet ? eau
Objet ? carte
Tu emportes 3 objets :
- casque
- eau
- carte
```

In [ ]:
%%exo 2.4
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("2.4")   # 💡 lance cette cellule si tu bloques

## 💪 Exercices de fin de bloc 2

### 🎯 Exercice 2.5 ⭐⭐⭐ — Les températures 🌡️

La liste `temperatures` est déjà écrite (ne la modifie pas). Construis une liste `positives` avec les températures **strictement positives**, affiche-la, puis affiche **combien** de températures sont négatives. Attention : 0 n'est ni positif ni négatif !

**Résultat attendu :**
```
[3, 8, 15]
Températures négatives : 3
```

In [ ]:
%%exo 2.5
temperatures = [-5, 3, -12, 8, 0, 15, -1]
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("2.5")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 2.6 ⭐⭐⭐ — La plus haute altitude ⛰️

La liste `altitudes` est déjà écrite (ne la modifie pas). Trouve et affiche la plus grande altitude **sans utiliser `max()`** ni trier la liste : avec une boucle !

**Résultat attendu :**
```
La plus haute altitude est 560
```

In [ ]:
%%exo 2.6
altitudes = [120, 340, 90, 560, 410]
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("2.6")   # 💡 lance cette cellule si tu bloques

## 🔥 Défi du bloc 2

### 🎯 Exercice 2.7 🔥 — La suite de Fibonacci

Dans la suite de Fibonacci, chaque nombre est la **somme des deux précédents** : 1, 1, 2, 3, 5, 8, 13… Demande `n`, construis une liste `fib` avec les `n` premiers nombres de la suite, puis affiche-la.

**Résultat attendu :**
```
Combien de termes ? 10
[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
```

In [ ]:
%%exo 2.7
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("2.7")   # 💡 lance cette cellule si tu bloques

---
## 📝 ÉVALUATION DU BLOC 2

### Partie 1 : QCM (10 points)

**Q1.** Avec `planetes = ["Mercure", "Vénus", "Terre"]`, que vaut `planetes[1]` ?

- **a)** "Mercure"
- **b)** "Vénus"
- **c)** "Terre"

In [ ]:
qcm("2.Q1", "?")   # ✍️ remplace ? par a, b ou c

**Q2.** Que vaut `len([4, 8, 15, 16])` ?

- **a)** 3
- **b)** 4
- **c)** 16

In [ ]:
qcm("2.Q2", "?")   # ✍️ remplace ? par a, b ou c

**Q3.** Comment ajouter 7 à la fin de la liste `L` ?

- **a)** L.add(7)
- **b)** L.append(7)
- **c)** L + 7

In [ ]:
qcm("2.Q3", "?")   # ✍️ remplace ? par a, b ou c

**Q4.** Avec `L = [1, 2, 3]`, que fait `print(L[3])` ?

- **a)** Ça affiche 3
- **b)** Ça affiche 0
- **c)** Ça provoque une erreur IndexError

In [ ]:
qcm("2.Q4", "?")   # ✍️ remplace ? par a, b ou c

**Q5.** Que donne `sorted([3, 1, 2])` ?

- **a)** [1, 2, 3]
- **b)** [3, 2, 1]
- **c)** [3, 1, 2]

In [ ]:
qcm("2.Q5", "?")   # ✍️ remplace ? par a, b ou c

### Partie 2 : exercices (10 points)

### 🎯 Exercice E2.1  — Les meilleurs scores 🏆

La liste `scores` est déjà écrite (ne la modifie pas). Affiche **combien** de scores sont supérieurs **ou égaux** à 1000.

**Résultat attendu :**
```
2 scores à 1000 ou plus
```

In [ ]:
%%exo E2.1
scores = [450, 1200, 300, 980, 1500]
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 🎯 Exercice E2.2  — Les multiples de 7

Demande un nombre `n`, construis la liste `multiples` des `n` premiers multiples de 7, puis affiche-la.

**Résultat attendu :**
```
n ? 5
[7, 14, 21, 28, 35]
```

In [ ]:
%%exo E2.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 📋 Ta note du bloc 2

In [ ]:
bilan(2)

---
# 🔤 BLOC 3 — Les chaînes de caractères

## 1. Une chaîne, c'est une liste de lettres

Une **chaîne de caractères** (en anglais *string*, `str`), c'est du texte entre guillemets. Elle se comporte **presque comme une liste de lettres** :

```python
mot = "fusee"
mot[0]      # "f"   (la 1re lettre, indice 0 !)
mot[-1]     # "e"   (la dernière)
len(mot)    # 5     (le nombre de lettres)
```

| Indice | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| `mot[...]` | f | u | s | e | e |

## 2. Parcourir lettre par lettre

```python
for lettre in "MARS":
    print(lettre)
```

## 3. Coller et répéter

```python
"fu" + "see"    # "fusee"  (on colle)
"Ha" * 3        # "HaHaHa" (on répète)
```

## 4. Les outils des chaînes

| Écriture | Résultat |
|---|---|
| `"terre".upper()` | `"TERRE"` (majuscules) |
| `"MARS".lower()` | `"mars"` (minuscules) |
| `"a b c".replace(" ", "-")` | `"a-b-c"` (remplacer) |
| `"terre".count("r")` | `2` (compter) |
| `"mit" in "admission au mit"` | `True` (est-ce dedans ?) |

## 5. Découper (le *slicing*)

```python
mot = "polytechnique"
mot[0:4]     # "poly"   (de l'indice 0 à 4 exclu, comme range !)
mot[4:]      # "technique"
mot[::-1]    # "euqinhcetylop"  (à l'envers !)
```

## 6. Les f-strings : mettre des variables dans du texte

Un `f` devant les guillemets, et les variables entre accolades `{ }` :

```python
prenom = "Ada"
age = 12
print(f"{prenom} a {age} ans et son prénom a {len(prenom)} lettres.")
```

## 7. ⚠️ On ne peut pas modifier une lettre

`mot[0] = "F"` provoque une **erreur** : une chaîne ne se modifie pas. Il faut **construire une nouvelle chaîne**, souvent dans une boucle :

```python
cri = ""
for lettre in "mars":
    cri = cri + lettre.upper() + "!"
print(cri)     # M!A!R!S!
```

In [ ]:
%%ralenti
cri = ""
for lettre in "mars":
    cri = cri + lettre.upper() + "!"
print(cri)

## 8. Exemples à lancer ▶

**Exemple A : explorer un mot.**

In [ ]:
mot = "polytechnique"
print("Première lettre :", mot[0])
print("Dernière lettre :", mot[-1])
print("Longueur :", len(mot))
print("Le début :", mot[0:4])
print("À l'envers :", mot[::-1])

**Exemple B : les outils.**

In [ ]:
phrase = "bonjour la terre"
print(phrase.upper())
print(phrase.replace("terre", "lune"))
print("Nombre de r :", phrase.count("r"))
print("lune" in phrase)

**Exemple C : une f-string.** Change le prénom !

In [ ]:
prenom = "Edouard"
print(f"Salut {prenom} ! Ton prénom a {len(prenom)} lettres et commence par {prenom[0]}.")

## 9. ⚠️ Les erreurs classiques

- Oublier que la 1re lettre est à l'indice **0**
- Oublier les parenthèses : `mot.upper` au lieu de `mot.upper()`
- Oublier le `f` devant une f-string → `{prenom}` s'affiche tel quel
- Vouloir modifier une lettre (`mot[0] = "F"`) → erreur

---
## ✍️ Exercices du bloc 3 — on commence doucement

### 🎯 Exercice 3.1 ⭐ — Bonjour ! 👋

Demande un prénom, puis affiche `Bonjour <prénom> ! Ton prénom a <N> lettres.`

**Résultat attendu :**
```
Ton prénom ? Edouard
Bonjour Edouard ! Ton prénom a 7 lettres.
```

In [ ]:
%%exo 3.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("3.1")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 3.2 ⭐ — Première et dernière lettre

Demande un mot, puis affiche sa **première** lettre, et sur la ligne suivante sa **dernière** lettre.

**Résultat attendu :**
```
Un mot ? fusee
f
e
```

In [ ]:
%%exo 3.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("3.2")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 3.3 ⭐⭐ — Compter les voyelles 🔤

Demande un mot, puis affiche combien il contient de voyelles (a, e, i, o, u, y), en minuscules.

**Résultat attendu :**
```
Ton mot ? fusee
Il y a 3 voyelles
```

In [ ]:
%%exo 3.3
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("3.3")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 3.4 ⭐⭐ — Le crieur spatial 📢

Demande une phrase. Affiche-la en **MAJUSCULES**, puis sur la ligne suivante, en majuscules avec des `_` à la place des espaces.

**Résultat attendu :**
```
Ta phrase ? bonjour la terre
BONJOUR LA TERRE
BONJOUR_LA_TERRE
```

In [ ]:
%%exo 3.4
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("3.4")   # 💡 lance cette cellule si tu bloques

## 💪 Exercices de fin de bloc 3

### 🎯 Exercice 3.5 ⭐⭐⭐ — Les palindromes 🔁

Un **palindrome** se lit pareil dans les deux sens : *kayak*, *radar*… Demande un mot et affiche `C'est un palindrome !` ou `Ce n'est pas un palindrome.`

**Résultat attendu :**
```
Un mot ? kayak
C'est un palindrome !
```

In [ ]:
%%exo 3.5
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("3.5")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 3.6 ⭐⭐⭐ — Le miroir 🪞

Demande un mot et affiche-le **à l'envers**, **sans** utiliser `[::-1]` ni `reversed` : construis-le avec une boucle !

**Résultat attendu :**
```
Un mot ? fusee
eesuf
```

In [ ]:
%%exo 3.6
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("3.6")   # 💡 lance cette cellule si tu bloques

## 🔥 Défi du bloc 3

### 🎯 Exercice 3.7 🔥 — Le code de César

Jules César chiffrait ses messages en **décalant chaque lettre** dans l'alphabet. Avec un décalage de 1 : a → b, b → c… et z → a.

Demande un message (en minuscules, sans accents) puis un décalage, et affiche le message chiffré. Les espaces restent des espaces.

**Résultat attendu :**
```
Message ? abc
Décalage ? 1
bcd
```

In [ ]:
%%exo 3.7
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("3.7")   # 💡 lance cette cellule si tu bloques

---
## 📝 ÉVALUATION DU BLOC 3

### Partie 1 : QCM (10 points)

**Q1.** Avec `mot = "fusee"`, que vaut `mot[0]` ?

- **a)** "f"
- **b)** "u"
- **c)** "e"

In [ ]:
qcm("3.Q1", "?")   # ✍️ remplace ? par a, b ou c

**Q2.** Que vaut `len("mit")` ?

- **a)** 2
- **b)** 3
- **c)** 4

In [ ]:
qcm("3.Q2", "?")   # ✍️ remplace ? par a, b ou c

**Q3.** Que donne `"Ha" * 3` ?

- **a)** "Ha3"
- **b)** "HaHaHa"
- **c)** Une erreur

In [ ]:
qcm("3.Q3", "?")   # ✍️ remplace ? par a, b ou c

**Q4.** Que donne `"Terre".upper()` ?

- **a)** "terre"
- **b)** "Terre"
- **c)** "TERRE"

In [ ]:
qcm("3.Q4", "?")   # ✍️ remplace ? par a, b ou c

**Q5.** Que donne `"polytechnique"[0:4]` ?

- **a)** "poly"
- **b)** "polyt"
- **c)** "olyt"

In [ ]:
qcm("3.Q5", "?")   # ✍️ remplace ? par a, b ou c

### Partie 2 : exercices (10 points)

### 🎯 Exercice E3.1  — Compter les e

Demande un mot, puis affiche combien de fois la lettre `e` apparaît dedans.

**Résultat attendu :**
```
Un mot ? fusee
2
```

In [ ]:
%%exo E3.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 🎯 Exercice E3.2  — Les tirets

Demande un mot et affiche ses lettres **séparées par des tirets**.

**Résultat attendu :**
```
Un mot ? mars
m-a-r-s
```

In [ ]:
%%exo E3.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 📋 Ta note du bloc 3

In [ ]:
bilan(3)

---
# 🏁 Fin du carnet !

Bravo d'être arrivé jusqu'ici 👏

- Lance `progression()` pour voir tout ce que tu as réussi.
- Quand tu as du réseau, lance `export()`, copie tout le texte et envoie-le à Claude avec le mot **« corrige »**. Je te ferai une correction détaillée et je mettrai à jour ton parcours sur GitHub.

In [ ]:
progression()

In [ ]:
export()

## 🧪 Bac à sable

Ces cellules sont pour toi : fais tous les essais que tu veux !

In [ ]:
# Essaie ce que tu veux ici 🚀